# B1: Linguistic Trajectory Analysis for Mental Health Monitoring

**Abstract.** Early detection of psychological distress from social media requires tracking how a user's language *evolves* over time, not just what they say at a single point. We demonstrate how ChronosVector enables continuous monitoring of linguistic embedding trajectories, using temporal velocity, drift attribution, change point detection, and stochastic characterization to identify early warning signals of mental health deterioration. Using real data from the eRisk/RSDD/CLPsych datasets, we show that trajectory-level analysis reveals dynamics invisible to single-snapshot classifiers.

**Keywords:** temporal embeddings, mental health, early detection, change point detection, drift analysis, social media

---

## 1. Introduction

Social media platforms generate continuous streams of text that, when embedded into vector space, form **temporal trajectories** — paths through high-dimensional space that encode how a user's language patterns evolve. Research has shown that linguistic markers precede clinical manifestations of depression, eating disorders, and self-harm (De Choudhury et al., 2013; Coppersmith et al., 2018; Couto et al., 2025).

Traditional approaches analyze snapshots: "is this post concerning?" ChronosVector enables a fundamentally different question: **"how is this user's language trajectory changing, and does the pattern match known deterioration dynamics?"**

In this tutorial we demonstrate:
1. **Trajectory ingestion** of real user embedding histories into CVX
2. **Velocity analysis** to quantify linguistic change rate
3. **Change point detection** (PELT) to identify regime transitions
4. **Drift attribution** to explain *what* changed (which semantic dimensions)
5. **Stochastic characterization** (Hurst exponent) to classify trajectory dynamics
6. **Prediction** to anticipate future trajectory states
7. **Temporal feature extraction** for downstream classification

## 2. Setup & Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
from pathlib import Path
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import time
import warnings
warnings.filterwarnings('ignore')

# ChronosVector Python bindings
import chronos_vector as cvx

# Visualization settings
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print(f'ChronosVector loaded')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')

In [ ]:
# ── Configuration ─────────────────────────────────────────────
# Point this to your pre-computed embeddings Parquet file.
# Generate it with: python scripts/generate_embeddings.py

DATA_PATH = Path('../data/embeddings/erisk_embeddings.parquet')

# Alternative datasets (uncomment one):
# DATA_PATH = Path('../data/embeddings/rsdd_embeddings.parquet')
# DATA_PATH = Path('../data/embeddings/clpsych_embeddings.parquet')

MIN_POSTS_PER_USER = 10   # Minimum posts to form a meaningful trajectory
MAX_USERS = 200            # Cap for computational tractability
TIMESTAMP_UNIT = 'days'    # Normalize timestamps to days for velocity interpretability

assert DATA_PATH.exists(), (
    f'Data not found at {DATA_PATH}. Please run:\n'
    f'  1. python scripts/preprocess_erisk.py --input data/erisk/raw/ --output data/erisk/unified.jsonl --labels data/erisk/golden_truth.txt\n'
    f'  2. python scripts/generate_embeddings.py --input data/erisk/unified.jsonl --output {DATA_PATH}'
)
print(f'Data source: {DATA_PATH}')

## 3. Data Loading

We load pre-computed sentence embeddings from the research dataset. Each row is a single post with its user ID, timestamp, depression/control label, and embedding vector.

In [ ]:
# Load embeddings
df = pd.read_parquet(DATA_PATH)

# Identify embedding columns
emb_cols = [c for c in df.columns if c.startswith('emb_')]
DIM = len(emb_cols)

print(f'Dataset summary:')
print(f'  Total posts: {len(df):,}')
print(f'  Users: {df["user_id"].nunique()}')
print(f'  Embedding dim: {DIM}')
print(f'  Date range: {df["timestamp"].min()} → {df["timestamp"].max()}')
print(f'  Label distribution:')
print(df.groupby('label')['user_id'].nunique().to_string())

# Filter users with sufficient posts
user_counts = df.groupby('user_id').size()
valid_users = user_counts[user_counts >= MIN_POSTS_PER_USER].index
df = df[df['user_id'].isin(valid_users)].copy()

# Cap number of users
if df['user_id'].nunique() > MAX_USERS:
    # Balanced sampling: equal depression/control
    users_per_label = MAX_USERS // 2
    dep_users = df[df['label'] == 'depression']['user_id'].unique()[:users_per_label]
    ctrl_users = df[df['label'] == 'control']['user_id'].unique()[:users_per_label]
    selected = np.concatenate([dep_users, ctrl_users])
    df = df[df['user_id'].isin(selected)].copy()

# Convert timestamps to day offsets (for interpretable velocity)
df['ts_days'] = (df['timestamp'] - df['timestamp'].min()).dt.total_seconds() / 86400
df['ts_us'] = (df['ts_days'] * 1_000_000).astype(int)  # microseconds for CVX

print(f'\nAfter filtering (≥{MIN_POSTS_PER_USER} posts, max {MAX_USERS} users):')
print(f'  Posts: {len(df):,}')
print(f'  Users: {df["user_id"].nunique()} ({df[df["label"]=="depression"]["user_id"].nunique()} depression, {df[df["label"]=="control"]["user_id"].nunique()} control)')

## 4. Ingestion into ChronosVector

We ingest all user trajectories into a CVX `TemporalIndex`. Each user's posts are ingested with their entity ID and timestamp.

In [ ]:
index = cvx.TemporalIndex(m=16, ef_construction=200, ef_search=100)

# Create numeric user IDs for CVX
user_id_map = {uid: i for i, uid in enumerate(df['user_id'].unique())}
user_labels = {user_id_map[uid]: label for uid, label in df.groupby('user_id')['label'].first().items()}

t0 = time.perf_counter()

for _, row in df.iterrows():
    eid = user_id_map[row['user_id']]
    ts = int(row['ts_us'])
    vec = row[emb_cols].values.astype(np.float32).tolist()
    index.insert(entity_id=eid, timestamp=ts, vector=vec)

elapsed = time.perf_counter() - t0

print(f'Ingested {len(df):,} posts in {elapsed:.2f}s')
print(f'Throughput: {len(df)/elapsed:,.0f} points/sec')
print(f'Index size: {len(index):,} vectors')

## 5. Trajectory Visualization

We project all embeddings to 2D via PCA and visualize trajectories colored by label. This gives a global view of how depression and control users occupy and move through the embedding space.

In [ ]:
# PCA projection
X = df[emb_cols].values
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)
df['pc1'] = X_2d[:, 0]
df['pc2'] = X_2d[:, 1]

print(f'PCA explained variance: {pca.explained_variance_ratio_[0]:.1%} + {pca.explained_variance_ratio_[1]:.1%} = {sum(pca.explained_variance_ratio_[:2]):.1%}')

# Global scatter
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_map = {'depression': '#e74c3c', 'control': '#2ecc71'}

for ax, label in zip(axes, ['control', 'depression']):
    subset = df[df['label'] == label]
    ax.scatter(subset['pc1'], subset['pc2'], c=colors_map[label], s=3, alpha=0.15)
    
    # Draw trajectories for 5 sample users
    sample_users = subset['user_id'].unique()[:5]
    for uid in sample_users:
        user_df = subset[subset['user_id'] == uid].sort_values('timestamp')
        ax.plot(user_df['pc1'], user_df['pc2'], alpha=0.5, linewidth=0.8)
        ax.scatter(user_df['pc1'].iloc[0], user_df['pc2'].iloc[0], c='green', s=40, marker='^', zorder=5)
        ax.scatter(user_df['pc1'].iloc[-1], user_df['pc2'].iloc[-1], c='black', s=40, marker='v', zorder=5)
    
    ax.set_title(f'{label.title()} Users (n={subset["user_id"].nunique()})', fontweight='bold')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')

fig.suptitle('Embedding Space: Depression vs Control Trajectories (PCA)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Velocity Analysis

The **embedding velocity** $\|\frac{\partial v}{\partial t}\|$ quantifies how fast a user's language is changing. We hypothesize that depression users show different velocity distributions than controls — either higher (rapid deterioration) or more variable (instability).

In [ ]:
# Compute mean velocity per user
velocity_results = []

for uid_str, group in df.groupby('user_id'):
    eid = user_id_map[uid_str]
    label = group['label'].iloc[0]
    traj_list = list(zip(
        group['ts_us'].tolist(),
        group[emb_cols].values.astype(np.float32).tolist()
    ))
    
    if len(traj_list) < 3:
        continue
    
    # Sample velocity at trajectory midpoint
    mid_ts = traj_list[len(traj_list) // 2][0]
    try:
        vel = cvx.velocity(traj_list, timestamp=mid_ts)
        vel_mag = float(np.linalg.norm(vel))
        velocity_results.append({'user_id': uid_str, 'label': label, 'velocity': vel_mag})
    except ValueError:
        pass

vel_df = pd.DataFrame(velocity_results)

# Statistical test
dep_vel = vel_df[vel_df['label'] == 'depression']['velocity']
ctrl_vel = vel_df[vel_df['label'] == 'control']['velocity']
t_stat, p_value = stats.mannwhitneyu(dep_vel, ctrl_vel, alternative='two-sided')
cohens_d = (dep_vel.mean() - ctrl_vel.mean()) / np.sqrt((dep_vel.std()**2 + ctrl_vel.std()**2) / 2)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(ctrl_vel, bins=30, alpha=0.6, color=colors_map['control'], label='Control', density=True)
axes[0].hist(dep_vel, bins=30, alpha=0.6, color=colors_map['depression'], label='Depression', density=True)
axes[0].axvline(ctrl_vel.mean(), color=colors_map['control'], linestyle='--', linewidth=2)
axes[0].axvline(dep_vel.mean(), color=colors_map['depression'], linestyle='--', linewidth=2)
axes[0].set_xlabel('|Velocity| (embedding units/day)')
axes[0].set_ylabel('Density')
axes[0].set_title('Velocity Distribution by Group', fontweight='bold')
axes[0].legend()

# Box plot
sns.boxplot(data=vel_df, x='label', y='velocity', palette=colors_map, ax=axes[1])
axes[1].set_title(f'Velocity by Group (p={p_value:.4f}, d={cohens_d:.3f})', fontweight='bold')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('|Velocity|')

plt.tight_layout()
plt.show()

print(f'Velocity Statistics:')
print(f'  Control:    mean={ctrl_vel.mean():.6f}  std={ctrl_vel.std():.6f}  n={len(ctrl_vel)}')
print(f'  Depression: mean={dep_vel.mean():.6f}  std={dep_vel.std():.6f}  n={len(dep_vel)}')
print(f'  Mann-Whitney U: p={p_value:.6f}')
print(f'  Cohen\'s d: {cohens_d:.4f}')

## 7. Change Point Detection

**PELT** (Pruned Exact Linear Time; Killick et al., 2012) detects structural breaks in embedding trajectories. We run it on all users and compare the number and severity of change points between depression and control groups.

In [ ]:
cpd_results = []

for uid_str, group in df.groupby('user_id'):
    eid = user_id_map[uid_str]
    label = group['label'].iloc[0]
    traj_list = list(zip(
        group['ts_us'].tolist(),
        group[emb_cols].values.astype(np.float32).tolist()
    ))
    
    if len(traj_list) < 10:
        continue
    
    cps = cvx.detect_changepoints(entity_id=eid, trajectory=traj_list)
    n_cps = len(cps)
    max_severity = max((sev for _, sev in cps), default=0.0)
    mean_severity = np.mean([sev for _, sev in cps]) if cps else 0.0
    
    cpd_results.append({
        'user_id': uid_str,
        'label': label,
        'n_changepoints': n_cps,
        'max_severity': max_severity,
        'mean_severity': mean_severity,
        'n_posts': len(traj_list),
        'cp_rate': n_cps / len(traj_list),  # normalized by trajectory length
    })

cpd_df = pd.DataFrame(cpd_results)

# Statistical tests
dep_cps = cpd_df[cpd_df['label'] == 'depression']
ctrl_cps = cpd_df[cpd_df['label'] == 'control']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Number of change points
sns.boxplot(data=cpd_df, x='label', y='n_changepoints', palette=colors_map, ax=axes[0])
u_stat, p_n = stats.mannwhitneyu(dep_cps['n_changepoints'], ctrl_cps['n_changepoints'])
axes[0].set_title(f'Number of Change Points (p={p_n:.4f})', fontweight='bold')

# Max severity
sns.boxplot(data=cpd_df, x='label', y='max_severity', palette=colors_map, ax=axes[1])
u_stat, p_s = stats.mannwhitneyu(dep_cps['max_severity'], ctrl_cps['max_severity'])
axes[1].set_title(f'Max CP Severity (p={p_s:.4f})', fontweight='bold')

# CP rate (normalized by trajectory length)
sns.boxplot(data=cpd_df, x='label', y='cp_rate', palette=colors_map, ax=axes[2])
u_stat, p_r = stats.mannwhitneyu(dep_cps['cp_rate'], ctrl_cps['cp_rate'])
axes[2].set_title(f'CP Rate (CPs/post, p={p_r:.4f})', fontweight='bold')

fig.suptitle('Change Point Detection: Depression vs Control', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Change Point Summary:')
print(cpd_df.groupby('label')[['n_changepoints', 'max_severity', 'cp_rate']].agg(['mean', 'std']).round(4))

## 8. Drift Attribution

For users with detected change points, we analyze **what changed** at the embedding level. This produces interpretable explanations of linguistic shifts.

In [ ]:
# Select a depression user with change points for detailed analysis
dep_with_cps = cpd_df[(cpd_df['label'] == 'depression') & (cpd_df['n_changepoints'] > 0)]

if not dep_with_cps.empty:
    example_uid = dep_with_cps.sort_values('max_severity', ascending=False).iloc[0]['user_id']
    example_group = df[df['user_id'] == example_uid].sort_values('timestamp')
    
    # Get first and last embedding
    v_first = example_group[emb_cols].iloc[0].values.astype(np.float32).tolist()
    v_last = example_group[emb_cols].iloc[-1].values.astype(np.float32).tolist()
    
    l2_mag, cosine_drift, top_dims = cvx.drift(v_first, v_last, top_n=10)
    
    print(f'Drift Analysis: Depression User {example_uid}')
    print(f'  Posts: {len(example_group)}, Days: {example_group["ts_days"].max() - example_group["ts_days"].min():.0f}')
    print(f'  L2 magnitude:  {l2_mag:.4f}')
    print(f'  Cosine drift:  {cosine_drift:.4f}')
    print(f'\n  Top contributing dimensions:')
    print(f'  {"Dim":>6} {"Direction":>10} {"|Change|":>10}')
    print('  ' + '-' * 28)
    for dim_idx, change in top_dims:
        direction = '↑' if (v_last[dim_idx] - v_first[dim_idx]) > 0 else '↓'
        print(f'  {dim_idx:>6} {direction:>10} {change:>10.4f}')
    
    # Heatmap of embedding evolution over time
    vecs = example_group[emb_cols].values
    # Show top 20 most variable dimensions
    dim_variance = np.var(vecs, axis=0)
    top_var_dims = np.argsort(dim_variance)[-20:][::-1]
    
    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(vecs[:, top_var_dims].T, aspect='auto', cmap='RdBu_r', interpolation='bilinear')
    ax.set_yticks(range(len(top_var_dims)))
    ax.set_yticklabels([f'dim_{d}' for d in top_var_dims], fontsize=8)
    ax.set_xlabel('Post index (time →)', fontsize=12)
    ax.set_title(f'Depression User {example_uid}: Top-20 Variable Dimensions Over Time', fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Embedding value', shrink=0.8)
    plt.tight_layout()
    plt.show()
else:
    print('No depression users with change points found — adjust PELT sensitivity or increase data.')

## 9. Stochastic Characterization

The **Hurst exponent** classifies trajectory dynamics:
- $H > 0.5$: Persistent/trending — deterioration likely to continue
- $H \approx 0.5$: Random walk — unpredictable
- $H < 0.5$: Anti-persistent — self-correcting

We test whether depression users show systematically different Hurst exponents.

In [ ]:
hurst_results = []

for uid_str, group in df.groupby('user_id'):
    label = group['label'].iloc[0]
    traj_list = list(zip(
        group['ts_us'].tolist(),
        group[emb_cols].values.astype(np.float32).tolist()
    ))
    
    if len(traj_list) < 20:
        continue
    
    try:
        h = cvx.hurst_exponent(traj_list)
        hurst_results.append({'user_id': uid_str, 'label': label, 'hurst': float(h)})
    except ValueError:
        pass

hurst_df = pd.DataFrame(hurst_results)

dep_h = hurst_df[hurst_df['label'] == 'depression']['hurst']
ctrl_h = hurst_df[hurst_df['label'] == 'control']['hurst']
u_stat, p_hurst = stats.mannwhitneyu(dep_h, ctrl_h, alternative='two-sided')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ctrl_h, bins=25, alpha=0.6, color=colors_map['control'], label='Control', density=True)
axes[0].hist(dep_h, bins=25, alpha=0.6, color=colors_map['depression'], label='Depression', density=True)
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='H=0.5 (random walk)')
axes[0].set_xlabel('Hurst Exponent')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Hurst Distribution (p={p_hurst:.4f})', fontweight='bold')
axes[0].legend()

sns.boxplot(data=hurst_df, x='label', y='hurst', palette=colors_map, ax=axes[1])
axes[1].axhline(0.5, color='black', linestyle='--', alpha=0.5)
axes[1].set_title('Hurst Exponent by Group', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Hurst Statistics:')
print(f'  Control:    mean={ctrl_h.mean():.4f}  std={ctrl_h.std():.4f}  n={len(ctrl_h)}')
print(f'  Depression: mean={dep_h.mean():.4f}  std={dep_h.std():.4f}  n={len(dep_h)}')
print(f'  Mann-Whitney U: p={p_hurst:.6f}')

## 10. Temporal Feature Extraction & Classification

CVX extracts a **fixed-size feature vector** from variable-length trajectories, encoding velocity, drift, volatility, change point count, and multi-scale dynamics. We use these as input to a Random Forest classifier to distinguish depression from control.

In [ ]:
# Extract temporal features for all users
feature_rows = []
feature_labels = []

for uid_str, group in df.groupby('user_id'):
    label = group['label'].iloc[0]
    traj_list = list(zip(
        group['ts_us'].tolist(),
        group[emb_cols].values.astype(np.float32).tolist()
    ))
    
    if len(traj_list) < 5:
        continue
    
    try:
        features = cvx.temporal_features(traj_list)
        feature_rows.append(features)
        feature_labels.append(1 if label == 'depression' else 0)
    except ValueError:
        pass

X_features = np.array(feature_rows)
y_labels = np.array(feature_labels)

print(f'Feature matrix: {X_features.shape}')
print(f'Labels: {sum(y_labels)} depression, {len(y_labels) - sum(y_labels)} control')

# Cross-validated classification
clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
scores = cross_val_score(clf, X_features, y_labels, cv=5, scoring='roc_auc')

print(f'\n5-Fold Cross-Validation (ROC-AUC):')
print(f'  Scores: {scores.round(4)}')
print(f'  Mean:   {scores.mean():.4f} ± {scores.std():.4f}')

# Feature importance
clf.fit(X_features, y_labels)
importances = clf.feature_importances_

fig, ax = plt.subplots(figsize=(14, 4))
top_n = 20
top_idx = np.argsort(importances)[-top_n:][::-1]
ax.bar(range(top_n), importances[top_idx], color='steelblue')
ax.set_xticks(range(top_n))
ax.set_xticklabels([f'f_{i}' for i in top_idx], rotation=45)
ax.set_xlabel('Feature Index')
ax.set_ylabel('Importance')
ax.set_title(f'Top-{top_n} Feature Importances (ROC-AUC={scores.mean():.3f})', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Prediction: Trajectory Forecasting

For users with sufficient history, we predict their embedding trajectory 7 days ahead and compare depression vs control forecast divergence.

In [ ]:
# Select one depression and one control user for comparison
dep_example = df[df['label'] == 'depression'].groupby('user_id').size().idxmax()
ctrl_example = df[df['label'] == 'control'].groupby('user_id').size().idxmax()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, uid_str, title_label in [(axes[0], ctrl_example, 'Control'), (axes[1], dep_example, 'Depression')]:
    eid = user_id_map[uid_str]
    group = df[df['user_id'] == uid_str].sort_values('timestamp')
    actual_vecs = group[emb_cols].values
    actual_days = group['ts_days'].values
    
    # Predict 7 days beyond last observation
    last_ts = int(group['ts_us'].max())
    pred_vecs = []
    pred_days = []
    for d in range(1, 8):
        target_ts = last_ts + d * 1_000_000  # 1 day in our units
        try:
            pred, method = index.predict(entity_id=eid, target_timestamp=target_ts)
            pred_vecs.append(pred)
            pred_days.append(actual_days[-1] + d)
        except ValueError:
            break
    
    if pred_vecs:
        pred_vecs = np.array(pred_vecs)
        
        # Project to PC1
        actual_pc1 = pca.transform(actual_vecs)[:, 0]
        pred_pc1 = pca.transform(pred_vecs)[:, 0]
        
        ax.plot(actual_days, actual_pc1, color=colors_map[group['label'].iloc[0]], linewidth=1.5, label='Observed')
        ax.plot(pred_days, pred_pc1, color='black', linewidth=2, linestyle='--', label='Predicted (7d)')
        ax.axvline(actual_days[-1], color='gray', linestyle=':', alpha=0.5)
    
    ax.set_title(f'{title_label} User (n={len(group)} posts)', fontweight='bold')
    ax.set_xlabel('Day')
    ax.set_ylabel('PC1')
    ax.legend(fontsize=9)

fig.suptitle('7-Day Trajectory Prediction: Control vs Depression', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 12. Summary & Clinical Implications

### Key Findings

In [ ]:
# Compile all results into a summary table
print('=' * 70)
print('SUMMARY OF FINDINGS')
print('=' * 70)
print(f'\nDataset: {DATA_PATH.name}')
print(f'Users: {df["user_id"].nunique()} ({df[df["label"]=="depression"]["user_id"].nunique()} dep, {df[df["label"]=="control"]["user_id"].nunique()} ctrl)')
print(f'Total posts: {len(df):,}')
print(f'Embedding dim: {DIM}')
print()
print(f'Velocity:')
print(f'  Depression mean: {dep_vel.mean():.6f}  Control mean: {ctrl_vel.mean():.6f}')
print(f'  p-value: {p_value:.6f}  Cohen\'s d: {cohens_d:.4f}')
print()
print(f'Change Points:')
print(f'  Depression mean: {dep_cps["n_changepoints"].mean():.2f}  Control mean: {ctrl_cps["n_changepoints"].mean():.2f}')
print(f'  p-value (n_cps): {p_n:.6f}')
print()
print(f'Hurst Exponent:')
print(f'  Depression mean: {dep_h.mean():.4f}  Control mean: {ctrl_h.mean():.4f}')
print(f'  p-value: {p_hurst:.6f}')
print()
print(f'Classification (CVX temporal features → Random Forest):')
print(f'  ROC-AUC: {scores.mean():.4f} ± {scores.std():.4f}')
print('=' * 70)

### Discussion

The results demonstrate that **temporal trajectory analysis** provides discriminative signals for mental health monitoring that go beyond single-snapshot classification:

- **Velocity** captures the *rate* of linguistic change, potentially flagging rapid deterioration
- **Change points** identify discrete *regime transitions* — the moments when language patterns shift
- **Hurst exponent** distinguishes *trending* (persistent deterioration) from *mean-reverting* (self-correcting) trajectories
- **Temporal features** provide a fixed-size representation that enables standard ML classification

### Limitations

- Self-reported labels may not reflect clinical diagnoses
- Embedding quality depends on the sentence transformer model
- Linear extrapolation is limited; Neural ODE backend is recommended for production
- Ethical considerations: automated mental health monitoring requires clinical oversight

### Clinical Pipeline

```
1. Embed user posts → sentence-transformers → D-dimensional vectors
2. Ingest into ChronosVector with user_id + timestamp
3. Continuous monitoring: velocity + BOCPD (real-time streaming)
4. On change point → drift attribution → clinician alert
5. Hurst classification → triage (trending → urgent, reverting → observe)
6. Prediction → anticipate trajectory 7–30 days ahead
```

---

## References

1. Couto, M. et al. (2025). Temporal word embeddings for psychological disorder early detection. *JHIR*.
2. Coppersmith, G. et al. (2018). NLP of social media as screening for suicide risk. *BMI Insights*.
3. De Choudhury, M. et al. (2013). Predicting depression via social media. *ICWSM*.
4. Killick, R. et al. (2012). Optimal detection of changepoints. *JASA*.
5. Chen, R.T.Q. et al. (2018). Neural ordinary differential equations. *NeurIPS*.
6. Hamilton, W.L. et al. (2016). Diachronic word embeddings reveal statistical laws of semantic change. *ACL*.
7. Yates, A. et al. (2017). Depression and self-harm risk assessment in online forums. *EMNLP*.